In [27]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler

In [28]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [29]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [30]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [31]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [32]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [33]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.00 0.95      0.07
LBP_003_PET                                                      0.00 0.98      0.03
LBP_012_CT                                                       0.00 0.97      0.04
LBP_012_PET                                                      0.00 0.98      0.03
LBP_021_CT                                                       0.00 0.98      0.02
LBP_021_PET                                                      0.00 0.95      0.08
LBP_030_CT                                                       0.00 1.00      0.01
LBP_030_PET       

In [34]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


In [35]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic      p  -log2(p)
LBP_003_CT                                                       0.16   0.69      0.53
LBP_003_PET                                                      0.13   0.72      0.48
LBP_012_CT                                                       0.03   0.86      0.21
LBP_012_PET                                                      0.11   0.75      0.42
LBP_021_CT                                                       2.78   0.10      3.39
LBP_021_PET                                                      0.02   0.90      0.15
LBP_030_CT                                                       0.06   0.81      0.30
LB

In [36]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['glcm_ClusterShade_d_1_PET_b2',
       'glrlm_LongRunLowGrayLevelEmphasis_CT_c16',
       'glszm_GrayLevelNonUniformity_PET_c04',
       'glszm_LargeAreaEmphasis_CT_c16',
       'glszm_LargeAreaLowGrayLevelEmphasis_CT_c16',
       'glszm_ZoneVariance_CT_c16'],
      dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [37]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [38]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [39]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [40]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,0.765178,0.610062,30.504395,50.002093,52.886671,58.000000,47.759816,58.864251,38067.208333,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,0.776540,0.504616,21.069386,41.753334,44.922155,44.294469,44.147480,48.723711,17870.791667,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,0.697164,0.478604,21.238275,44.375483,48.466483,45.891176,37.656341,48.969378,17426.500000,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,0.574636,0.446059,20.570459,46.115989,42.544095,47.507894,33.837849,55.226805,12463.791667,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,0.633419,0.480378,26.130155,54.394967,66.030296,56.320511,45.398238,67.089492,28300.125000,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,0.882411,0.577884,19.774388,34.218615,41.785165,40.261644,35.805028,43.520110,12822.458333,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,0.535802,0.455642,23.259099,51.046869,51.264022,51.264022,33.600595,52.440442,18368.291667,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,0.716610,0.631485,31.838169,50.417953,46.324939,56.035703,57.070132,57.671483,35384.541667,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,0.665145,0.628338,28.213278,44.901412,50.289164,50.596443,35.777088,51.478151,26180.208333,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [41]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [42]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [43]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [44]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 374)
y_train:  (139,)


In [46]:
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [47]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 376)

In [48]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data[vif_data['VIF'] > 100000]

,feature,VIF
0,shape_Elongation,2.096355e+11
1,shape_Flatness,3.609232e+11
2,shape_LeastAxisLength,2.691126e+12
3,shape_MajorAxisLength,7.524811e+12
4,shape_Maximum2DDiameterColumn,2.124339e+13
...,...,...
369,LBP_111_PET,5.922282e+11
370,LBP_120_PET,3.305758e+11
371,LBP_201_PET,1.508996e+12
372,LBP_210_PET,1.369708e+12


# Standardization

In [49]:
original_X = X.copy()

In [50]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

# Standardize non-categorical and then concat with the categorical
scaler = StandardScaler()  
X_std = scaler.fit_transform(X)
X_std = pd.DataFrame(X_std, columns=X_columns, index=X_index)

In [51]:
# Standardize the numeric part 
X_MAASTRO_column = X_MAASTRO.columns
X_MAASTRO_index = X_MAASTRO.index

X_MAASTRO_std = scaler.transform(X_MAASTRO)

# Change the standardized part into a dataframe 
X_MAASTRO_std = pd.DataFrame(X_MAASTRO_std, columns=X_MAASTRO_column, index=X_MAASTRO_index)

In [52]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = X_MAASTRO_std 

# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [53]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:40:48,895] A new study created in memory with name: no-name-40d4a3c1-121a-4576-a354-100423593b61
python(82517) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2024-04-16 01:40:51,155] Trial 0 failed with parameters: {} because of the following error: LinAlgError('Matrix is singular.').
Traceback (most recent call last):
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/64/jkqp6xyx2hj50dmd2pqfm3780000gn/T/ipykernel_81654/4028431732.py", line 62, in objective
    model.fit(X_train_std, y_train)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/sksurv/linear_model/coxph.py", line 449, in fit
    delta = solve(
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 220, in solve
    _solve_check(n, info)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 29, in _solve_check
    raise LinAlgError('Matrix is singular.')
numpy.linalg.LinAlgError: Matrix is singular.
[W 2024-04-16 01:40:51,178] Trial 0 fai

LinAlgError: Matrix is singular.

In [54]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

ValueError: No trials are completed yet.

In [55]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

ValueError: No trials are completed yet.

#### Test

In [56]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [57]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

LinAlgError: Matrix is singular.

In [58]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [59]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [60]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 09:49:41,247] A new study created in memory with name: no-name-1c0b5bf8-4789-415d-bef5-63c0383f64cf


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6060606060606061
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7303921568627451
Fold 4 C-index: 0.7151898734177216


[I 2024-04-16 09:49:42,050] A new study created in memory with name: no-name-e60780e9-76d8-4aa9-840e-96f4a5091e04


Fold 5 C-index: 0.6971830985915493
[I 2024-04-16 09:49:42,040] Trial 0 finished with value: 0.6979794327008101 and parameters: {}. Best is trial 0 with value: 0.6979794327008101.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6979794327008101], datetime_start=datetime.datetime(2024, 4, 16, 9, 49, 41, 346876), datetime_complete=datetime.datetime(2024, 4, 16, 9, 49, 42, 38938), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6979794327008101


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651743727053
Fold 2 IBS: 0.22157791159752618
Fold 3 IBS: 0.20453593920513286
Fold 4 IBS: 0.22473803623356947
Fold 5 IBS: 0.21812430785364773
[I 2024-04-16 09:49:43,081] Trial 0 finished with value: 0.21659054246542936 and parameters: {}. Best is trial 0 with value: 0.21659054246542936.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054246542936], datetime_start=datetime.datetime(2024, 4, 16, 9, 49, 42, 198811), datetime_complete=datetime.datetime(2024, 4, 16, 9, 49, 43, 80688), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054246542936


In [61]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [62]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.698
train_ibs:  0.217


#### Test

In [63]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [64]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.586


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [65]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [66]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 09:49:43,538] A new study created in memory with name: no-name-6bf5df5a-4149-4905-9669-7f50ef02adfa


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6060606060606061
Fold 2 C-index: 0.48214285714285715
Fold 3 C-index: 0.46078431372549017
Fold 4 C-index: 0.5738396624472574


[I 2024-04-16 09:49:48,577] A new study created in memory with name: no-name-2871781a-bef6-4cb0-8807-43c3cf535e81


Fold 5 C-index: 0.5821596244131455
[I 2024-04-16 09:49:48,557] Trial 0 finished with value: 0.5409974127578712 and parameters: {}. Best is trial 0 with value: 0.5409974127578712.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5409974127578712], datetime_start=datetime.datetime(2024, 4, 16, 9, 49, 43, 602182), datetime_complete=datetime.datetime(2024, 4, 16, 9, 49, 48, 556718), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5409974127578712


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.3951989555259915
Fold 2 IBS: 0.4946714020030033
Fold 3 IBS: 0.42816946690567437
Fold 4 IBS: 0.33291518222236205
Fold 5 IBS: 0.3096472267208097
[I 2024-04-16 09:49:55,048] Trial 0 finished with value: 0.39212044667556817 and parameters: {}. Best is trial 0 with value: 0.39212044667556817.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.39212044667556817], datetime_start=datetime.datetime(2024, 4, 16, 9, 49, 48, 620984), datetime_complete=datetime.datetime(2024, 4, 16, 9, 49, 55, 47450), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.39212044667556817


In [67]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [68]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.541
train_ibs:  0.392


#### Test

In [69]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [70]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.5


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.43


In [71]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [72]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 09:49:57,567] A new study created in memory with name: no-name-7e57dc36-46bd-46d3-98f0-5192775006d0


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.5
Fold 3 C-index: 0.4950980392156863
Fold 4 C-index: 0.6160337552742616
Fold 5 C-index: 0.6291079812206573
[I 2024-04-16 09:50:01,236] Trial 0 finished with value: 0.5666626737568397 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.5666626737568397.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.5491071428571429
Fold 3 C-index: 0.5980392156862745
Fold 4 C-index: 0.6371308016877637
Fold 5 C-index: 0.6431924882629108
[I 2024-04-16 09:50:03,571] Trial 1 finished with value: 0.6058402500451388 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6058402500451388.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.5491071428571429
Fold 3 C-index: 0.6274509803921569
Fold 4 C-index: 0.6455696202531646
Fold 5 C-index: 0.6431924882629108
[I 2024-04-16 09:50:06,023] Trial 2 finished with value: 0.6134103666993953 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 

Fold 1 C-index: 0.6147186147186147
Fold 2 C-index: 0.5848214285714286
Fold 3 C-index: 0.6225490196078431
Fold 4 C-index: 0.6540084388185654
Fold 5 C-index: 0.6384976525821596
[I 2024-04-16 09:50:58,704] Trial 24 finished with value: 0.6229190308597222 and parameters: {'l1_ratio': 0.13208343776906337}. Best is trial 12 with value: 0.7073708888407955.
Fold 1 C-index: 0.5974025974025974
Fold 2 C-index: 0.5401785714285714
Fold 3 C-index: 0.5686274509803921
Fold 4 C-index: 0.6371308016877637
Fold 5 C-index: 0.6384976525821596
[I 2024-04-16 09:51:01,031] Trial 25 finished with value: 0.5963674148162968 and parameters: {'l1_ratio': 0.3370963772856702}. Best is trial 12 with value: 0.7073708888407955.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.5580357142857143
Fold 3 C-index: 0.6176470588235294
Fold 4 C-index: 0.6455696202531646
Fold 5 C-index: 0.6291079812206573
[I 2024-04-16 09:51:03,388] Trial 26 finished with value: 0.6104183952629334 and parameters: {'l1_ratio': 0.20688108474704

Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7205882352941176
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.676056338028169
[I 2024-04-16 09:51:50,616] Trial 48 finished with value: 0.7008616013870175 and parameters: {'l1_ratio': 0.036675112124032225}. Best is trial 12 with value: 0.7073708888407955.
Fold 1 C-index: 0.5974025974025974
Fold 2 C-index: 0.5357142857142857
Fold 3 C-index: 0.5637254901960784
Fold 4 C-index: 0.6371308016877637
Fold 5 C-index: 0.6384976525821596
[I 2024-04-16 09:51:53,385] Trial 49 finished with value: 0.594494165516577 and parameters: {'l1_ratio': 0.3467452999146196}. Best is trial 12 with value: 0.7073708888407955.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.5267857142857143
Fold 3 C-index: 0.5294117647058824
Fold 4 C-index: 0.6329113924050633
Fold 5 C-index: 0.6384976525821596
[I 2024-04-16 09:51:55,340] Trial 50 finished with value: 0.5841360234104825 and parameters: {'l1_ratio': 0.47241766561900594}. Best is tr

...

Fold 1 IBS: 0.3553857989151826
Fold 2 IBS: 0.4113662233415645
Fold 3 IBS: 0.39839830737353576
Fold 4 IBS: 0.2999338715886926
Fold 5 IBS: 0.2525302607069104
[I 2024-04-16 09:53:05,007] Trial 0 finished with value: 0.34352289238517714 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.34352289238517714.
Fold 1 IBS: 0.3403896464980156
Fold 2 IBS: 0.34222993397115575
Fold 3 IBS: 0.2895050096292098
Fold 4 IBS: 0.2615594971797559
Fold 5 IBS: 0.22664790373670732
[I 2024-04-16 09:53:06,658] Trial 1 finished with value: 0.29206639820296887 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.29206639820296887.
Fold 1 IBS: 0.3319522647575167
Fold 2 IBS: 0.33499782694450664
Fold 3 IBS: 0.274212031131214
Fold 4 IBS: 0.25714328930631974
Fold 5 IBS: 0.22706925204032463
[I 2024-04-16 09:53:08,138] Trial 2 finished with value: 0.28507493283597635 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.28507493283597635.


Fold 1 IBS: 0.3192240074992903
Fold 2 IBS: 0.32344762238161084
Fold 3 IBS: 0.2590595935986579
Fold 4 IBS: 0.2526430043357355
Fold 5 IBS: 0.23033566460273056
[I 2024-04-16 09:53:42,315] Trial 25 finished with value: 0.276941978483605 and parameters: {'l1_ratio': 0.16690584651330848}. Best is trial 11 with value: 0.2158776110721526.
Fold 1 IBS: 0.3747824246078294
Fold 2 IBS: 0.4360107780095121
Fold 3 IBS: 0.40648779968059867
Fold 4 IBS: 0.31897369761290323
Fold 5 IBS: 0.2660194668146372
[I 2024-04-16 09:53:44,413] Trial 26 finished with value: 0.3604548333450961 and parameters: {'l1_ratio': 0.8401513352750697}. Best is trial 11 with value: 0.2158776110721526.
Fold 1 IBS: 0.32520772497458367
Fold 2 IBS: 0.32868353816868084
Fold 3 IBS: 0.26491020503681695
Fold 4 IBS: 0.2545661773871685
Fold 5 IBS: 0.22863061131601403
[I 2024-04-16 09:53:45,765] Trial 27 finished with value: 0.2803996513766528 and parameters: {'l1_ratio': 0.19058100336990122}. Best is trial 11 with value: 0.2158776110721526

Fold 1 IBS: 0.31156633973707326
Fold 2 IBS: 0.3163865528172471
Fold 3 IBS: 0.2516857469278973
Fold 4 IBS: 0.2507081100985178
Fold 5 IBS: 0.2325148444425551
[I 2024-04-16 09:54:17,168] Trial 50 finished with value: 0.2725723188046581 and parameters: {'l1_ratio': 0.14081250399003042}. Best is trial 11 with value: 0.2158776110721526.
Fold 1 IBS: 0.2973879286502772
Fold 2 IBS: 0.29867786958963066
Fold 3 IBS: 0.23332104975980586
Fold 4 IBS: 0.24660822050948716
Fold 5 IBS: 0.2349169293427148
[I 2024-04-16 09:54:18,769] Trial 51 finished with value: 0.26218239957038314 and parameters: {'l1_ratio': 0.09420281977278813}. Best is trial 11 with value: 0.2158776110721526.
Fold 1 IBS: 0.26528647509818026
Fold 2 IBS: 0.22019990932883693
Fold 3 IBS: 0.18930339998606083
Fold 4 IBS: 0.2234937826734629
Fold 5 IBS: 0.22756144788673846
[I 2024-04-16 09:54:20,011] Trial 52 finished with value: 0.22516900299465586 and parameters: {'l1_ratio': 0.026036492952643504}. Best is trial 11 with value: 0.21587761107

Fold 1 IBS: 0.2820537811446324
Fold 2 IBS: 0.27281077815346333
Fold 3 IBS: 0.2109882464190534
Fold 4 IBS: 0.2413619551337872
Fold 5 IBS: 0.2331403453989755
[I 2024-04-16 09:54:52,893] Trial 75 finished with value: 0.24807102124998237 and parameters: {'l1_ratio': 0.05358076471620459}. Best is trial 70 with value: 0.21261233330990087.
Fold 1 IBS: 0.30453887168064997
Fold 2 IBS: 0.30815924821000296
Fold 3 IBS: 0.2433574571045309
Fold 4 IBS: 0.24874086065274206
Fold 5 IBS: 0.2340943831783615
[I 2024-04-16 09:54:54,710] Trial 76 finished with value: 0.2677781641652575 and parameters: {'l1_ratio': 0.11688592011583543}. Best is trial 70 with value: 0.21261233330990087.
Fold 1 IBS: 0.2975855759165154
Fold 2 IBS: 0.2989747817947932
Fold 3 IBS: 0.2336129432464095
Fold 4 IBS: 0.24665078013932817
Fold 5 IBS: 0.23491117225580493
[I 2024-04-16 09:54:56,191] Trial 77 finished with value: 0.26234705067057024 and parameters: {'l1_ratio': 0.09484096699800859}. Best is trial 70 with value: 0.212612333309

Fold 5 IBS: 0.2240565139267466
[I 2024-04-16 09:55:22,498] Trial 99 finished with value: 0.21250990151838317 and parameters: {'l1_ratio': 0.01697082373681321}. Best is trial 81 with value: 0.2118809212031346.


* Best trial for IBS: 
 FrozenTrial(number=81, state=TrialState.COMPLETE, values=[0.2118809212031346], datetime_start=datetime.datetime(2024, 4, 16, 9, 55, 1, 530603), datetime_complete=datetime.datetime(2024, 4, 16, 9, 55, 2, 367003), params={'l1_ratio': 0.014604550367388038}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=81, value=None)


* Best Score for IBS: 
 0.2118809212031346


In [73]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [74]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.708
train_ibs:  0.212


#### Test

In [75]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [76]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.025164283254476998)

test_cindex : 0.586


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.014604550367388038)

test_ibs:  0.221


In [77]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [78]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 09:55:23,016] A new study created in memory with name: no-name-fb9a45c8-a99f-45d8-9aa6-fcb72591811a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.7058823529411765
Fold 4 C-index: 0.6835443037974683
Fold 5 C-index: 0.7183098591549296
[I 2024-04-16 09:55:57,908] Trial 0 finished with value: 0.7030408096722214 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7030408096722214.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.6421568627450981
Fold 4 C-index: 0.7426160337552743
Fold 5 C-index: 0.6619718309859155
[I 2024-04-16 09:56:04,527] Trial 1 finished with value: 0.6866757853240973 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.7464788732394366
[I 2024-04-16 09:58:12,954] Trial 15 finished with value: 0.7343450964479536 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 2, 'max_depth': 17, 'n_estimators': 491, 'oob_score': True, 'max_samples': 0.25141113328809755, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.08807762154599924, 'warm_start': True}. Best is trial 14 with value: 0.7602465352824544.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 09:58:14,461] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 16, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.14975474925617, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.08683497083568477, 'warm_start': True}. Best is trial 14 with value: 0.7602465352824544.
Fold 1 C-inde

Fold 1 C-index: 0.6103896103896104
Fold 2 C-index: 0.875
Fold 3 C-index: 0.8431372549019608
Fold 4 C-index: 0.9029535864978903
Fold 5 C-index: 0.8826291079812206
[I 2024-04-16 09:59:17,279] Trial 31 finished with value: 0.8228219119541365 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 20, 'min_samples_leaf': 3, 'max_depth': 8, 'n_estimators': 447, 'oob_score': True, 'max_samples': 0.5119028984984925, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.04799860054302019, 'warm_start': True}. Best is trial 25 with value: 0.8467871740018396.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.8883928571428571
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.919831223628692
Fold 5 C-index: 0.8826291079812206
[I 2024-04-16 09:59:21,815] Trial 32 finished with value: 0.8283539838162529 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 6, 'n_estimators': 430, 'oob_score': True, 'max_samples': 0.5077512231481354, 'max_features

...

Fold 1 IBS: 0.20878411136352604
Fold 2 IBS: 0.18596824822942906
Fold 3 IBS: 0.19066800479976556
Fold 4 IBS: 0.17194213397317673
Fold 5 IBS: 0.20965199730588724
[I 2024-04-16 10:08:05,267] Trial 32 finished with value: 0.1934028991343569 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 6, 'n_estimators': 272, 'oob_score': True, 'max_samples': 0.7212112540749575, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.10884764482583958}. Best is trial 0 with value: 0.18893690419369852.
Fold 1 IBS: 0.26672933578367825
Fold 2 IBS: 0.17260981802328967
Fold 3 IBS: 0.218462235370602
Fold 4 IBS: 0.17492701943122574
Fold 5 IBS: 0.1712325610179695
[I 2024-04-16 10:08:09,581] Trial 33 finished with value: 0.20079219392535302 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 19, 'oob_score': True, 'max_samples': 0.6049706297065969, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.2015902982015954
Fold 2 IBS: 0.1857104387603733
Fold 3 IBS: 0.1790267886123672
Fold 4 IBS: 0.19557127495348786
Fold 5 IBS: 0.20398058535823538
[I 2024-04-16 10:10:02,715] Trial 48 finished with value: 0.19317587717721182 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 2, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 40, 'oob_score': False, 'max_samples': 0.5260405815006272, 'max_features': None, 'min_weight_fraction_leaf': 0.21390971396804748}. Best is trial 39 with value: 0.18799935749136473.
Fold 1 IBS: 0.2145391454105467
Fold 2 IBS: 0.22377474389972024
Fold 3 IBS: 0.20445767011409766
Fold 4 IBS: 0.22686036777266955
Fold 5 IBS: 0.2243610480605514
[I 2024-04-16 10:10:03,070] Trial 49 finished with value: 0.21879859505151708 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 6, 'min_samples_leaf': 20, 'max_depth': 16, 'n_estimators': 11, 'oob_score': False, 'max_samples': 0.26830717105159463, 'max_features': None, 'min_weight_fraction_leaf':

Fold 1 IBS: 0.1928095274883131
Fold 2 IBS: 0.1829210242425621
Fold 3 IBS: 0.19880360343847653
Fold 4 IBS: 0.1863866229226835
Fold 5 IBS: 0.1951582078859148
[I 2024-04-16 10:12:26,805] Trial 64 finished with value: 0.19121579719558998 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 19, 'min_samples_leaf': 7, 'max_depth': 17, 'n_estimators': 188, 'oob_score': False, 'max_samples': 0.775924036717329, 'max_features': None, 'min_weight_fraction_leaf': 0.20625622507811242}. Best is trial 39 with value: 0.18799935749136473.
Fold 1 IBS: 0.19101081770652317
Fold 2 IBS: 0.18371867374090417
Fold 3 IBS: 0.1824200640830827
Fold 4 IBS: 0.19925148398280748
Fold 5 IBS: 0.19796558416349588
[I 2024-04-16 10:12:36,770] Trial 65 finished with value: 0.19087332473536267 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 16, 'min_samples_leaf': 18, 'max_depth': 12, 'n_estimators': 256, 'oob_score': False, 'max_samples': 0.6380685376962676, 'max_features': None, 'min_weight_fraction_leaf

Fold 5 IBS: 0.19832262248550672
[I 2024-04-16 10:15:08,851] Trial 79 finished with value: 0.19171047506200603 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 10, 'min_samples_leaf': 14, 'max_depth': 12, 'n_estimators': 25, 'oob_score': False, 'max_samples': 0.4577545127391126, 'max_features': None, 'min_weight_fraction_leaf': 0.09781938022887349}. Best is trial 39 with value: 0.18799935749136473.
Fold 1 IBS: 0.19426839896125792
Fold 2 IBS: 0.17981710691108702
Fold 3 IBS: 0.17861065931527556
Fold 4 IBS: 0.19911223696785046
Fold 5 IBS: 0.2081704718696194
[I 2024-04-16 10:15:11,480] Trial 80 finished with value: 0.19199577480501806 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 16, 'max_depth': 13, 'n_estimators': 99, 'oob_score': False, 'max_samples': 0.40696501409993335, 'max_features': None, 'min_weight_fraction_leaf': 0.05418386307803515}. Best is trial 39 with value: 0.18799935749136473.
Fold 1 IBS: 0.1928921274877498
Fold 2 IBS: 0.179

Fold 1 IBS: 0.19946297870294474
Fold 2 IBS: 0.1787195203639327
Fold 3 IBS: 0.18978142216242233
Fold 4 IBS: 0.19366785888566598
Fold 5 IBS: 0.20885493662651164
[I 2024-04-16 10:15:47,097] Trial 95 finished with value: 0.19409734334829548 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 13, 'min_samples_leaf': 18, 'max_depth': 16, 'n_estimators': 55, 'oob_score': False, 'max_samples': 0.43465916069396443, 'max_features': None, 'min_weight_fraction_leaf': 0.14266992337047604}. Best is trial 88 with value: 0.18256829972921168.
Fold 1 IBS: 0.21394879098507638
Fold 2 IBS: 0.22071767868681683
Fold 3 IBS: 0.204663570980379
Fold 4 IBS: 0.22456834280613222
Fold 5 IBS: 0.21820813733001868
[I 2024-04-16 10:15:47,996] Trial 96 finished with value: 0.21642130415768462 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 11, 'min_samples_leaf': 14, 'max_depth': 19, 'n_estimators': 91, 'oob_score': False, 'max_samples': 0.2533979604399289, 'max_features': None, 'min_weight_fraction_l

In [79]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [80]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.871
train_ibs:  0.183


#### Test

In [81]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

In [82]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=8, max_features='auto', max_leaf_nodes=17,
                     max_samples=0.9776940371468902, min_samples_split=9,
                     min_weight_fraction_leaf=0.03508377856391107,
                     n_estimators=19, random_state=123, warm_start=True)

test_cindex:  0.628


RandomSurvivalForest(max_depth=17, max_features=None, max_leaf_nodes=9,
                     max_samples=0.4387811941970341, min_samples_leaf=15,
                     min_samples_split=14,
                     min_weight_fraction_leaf=0.11484074874869586,
                     n_estimators=5, random_state=123)

test_ibs:  0.226


In [83]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [84]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [85]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 10:15:51,192] A new study created in memory with name: no-name-aff9ee24-a011-45da-9bf0-d87d9261b0bb


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6147186147186147
Fold 2 C-index: 0.8526785714285714
Fold 3 C-index: 0.75
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.755868544600939
[I 2024-04-16 10:15:52,215] Trial 0 finished with value: 0.7473957621833802 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7473957621833802.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:15:54,425] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. Best is tria

Fold 5 C-index: 0.8826291079812206
[I 2024-04-16 10:16:21,413] Trial 15 finished with value: 0.8175678569374233 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 9, 'min_samples_leaf': 6, 'max_depth': 11, 'n_estimators': 362, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.9163782618183121, 'min_weight_fraction_leaf': 0.07436726243220565}. Best is trial 15 with value: 0.8175678569374233.
Fold 1 C-index: 0.6406926406926406
Fold 2 C-index: 0.8214285714285714
Fold 3 C-index: 0.7303921568627451
Fold 4 C-index: 0.70042194092827
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 10:16:22,514] Trial 16 finished with value: 0.7175541981326802 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 9, 'min_samples_leaf': 6, 'max_depth': 9, 'n_estimators': 277, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.89699542693011, 'min_weight_fraction_leaf': 0.35612645124522524}. Best is trial 15 with value: 0.8175678569374233.
F

Fold 1 C-index: 0.5887445887445888
Fold 2 C-index: 0.8258928571428571
Fold 3 C-index: 0.696078431372549
Fold 4 C-index: 0.7130801687763713
Fold 5 C-index: 0.6525821596244131
[I 2024-04-16 10:17:01,698] Trial 30 finished with value: 0.6952756411321559 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 9, 'max_depth': 9, 'n_estimators': 394, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.8382504072008882, 'min_weight_fraction_leaf': 0.21007556306018008}. Best is trial 25 with value: 0.8353010848093593.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.8945147679324894
Fold 5 C-index: 0.8685446009389671
[I 2024-04-16 10:17:03,332] Trial 31 finished with value: 0.8109658972018442 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 11, 'min_samples_leaf': 7, 'max_depth': 3, 'n_estimators': 404, 'oob_score': False, 'warm_start': True, 'max_features': 

...

Fold 1 IBS: 0.2081043148093001
Fold 2 IBS: 0.20420509768667056
Fold 3 IBS: 0.1962340623978483
Fold 4 IBS: 0.2096506990203624
Fold 5 IBS: 0.21353357787911675
[I 2024-04-16 10:20:43,284] Trial 30 finished with value: 0.20634555035865962 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 4, 'min_samples_leaf': 17, 'max_depth': 12, 'n_estimators': 381, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.7314863609037403, 'min_weight_fraction_leaf': 0.10563608457720963}. Best is trial 27 with value: 0.18709501535488485.
Fold 1 IBS: 0.19730717896052297
Fold 2 IBS: 0.18529441321477938
Fold 3 IBS: 0.18533366129064377
Fold 4 IBS: 0.18248960193133745
Fold 5 IBS: 0.20221423911324402
[I 2024-04-16 10:20:49,252] Trial 31 finished with value: 0.19052781890210552 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 7, 'min_samples_leaf': 13, 'max_depth': 12, 'n_estimators': 392, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0

Fold 1 IBS: 0.2057886310768082
Fold 2 IBS: 0.19693079192071614
Fold 3 IBS: 0.19129734861332404
Fold 4 IBS: 0.19629616500268943
Fold 5 IBS: 0.21038967978117093
[I 2024-04-16 10:22:00,712] Trial 45 finished with value: 0.20014052327894177 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 3, 'min_samples_leaf': 11, 'max_depth': 15, 'n_estimators': 486, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.9606319416987669, 'min_weight_fraction_leaf': 0.06471432252401287}. Best is trial 27 with value: 0.18709501535488485.
Fold 1 IBS: 0.20934463006342957
Fold 2 IBS: 0.16626348855382878
Fold 3 IBS: 0.19117275286807775
Fold 4 IBS: 0.1653610699678058
Fold 5 IBS: 0.19753335331998537
[I 2024-04-16 10:22:05,542] Trial 46 finished with value: 0.18593505895462542 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 3, 'min_samples_leaf': 5, 'max_depth': 19, 'n_estimators': 272, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0

Fold 1 IBS: 0.207285461038795
Fold 2 IBS: 0.1927181173372502
Fold 3 IBS: 0.1875847073262801
Fold 4 IBS: 0.18626542533097468
Fold 5 IBS: 0.21021231166082874
[I 2024-04-16 10:23:01,517] Trial 60 finished with value: 0.19681320453882573 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 16, 'n_estimators': 303, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.9034006727918623, 'min_weight_fraction_leaf': 0.05252660219522337}. Best is trial 46 with value: 0.18593505895462542.
Fold 1 IBS: 0.22222320857283578
Fold 2 IBS: 0.16627488261932677
Fold 3 IBS: 0.1907907801291295
Fold 4 IBS: 0.15665311269211
Fold 5 IBS: 0.19448240192170554
[I 2024-04-16 10:23:07,584] Trial 61 finished with value: 0.18608487718702152 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 4, 'min_samples_leaf': 4, 'max_depth': 18, 'n_estimators': 274, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.93657

Fold 1 IBS: 0.22613938863893251
Fold 2 IBS: 0.17078566610790855
Fold 3 IBS: 0.1918060554350325
Fold 4 IBS: 0.1609525872259648
Fold 5 IBS: 0.19576638517676706
[I 2024-04-16 10:24:45,614] Trial 75 finished with value: 0.1890900165169211 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 19, 'n_estimators': 376, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6854858143225422, 'min_weight_fraction_leaf': 0.014781978409836638}. Best is trial 46 with value: 0.18593505895462542.
Fold 1 IBS: 0.20539145272671636
Fold 2 IBS: 0.19530699928990344
Fold 3 IBS: 0.18948528388235533
Fold 4 IBS: 0.19829570272263905
Fold 5 IBS: 0.21090398924576984
[I 2024-04-16 10:24:49,816] Trial 76 finished with value: 0.1998766855734768 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 4, 'min_samples_leaf': 7, 'max_depth': 19, 'n_estimators': 279, 'oob_score': True, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.86

Fold 1 IBS: 0.20699759070252
Fold 2 IBS: 0.18410685277890454
Fold 3 IBS: 0.18353461706009355
Fold 4 IBS: 0.17619715129676206
Fold 5 IBS: 0.20499796617822919
[I 2024-04-16 10:25:48,981] Trial 90 finished with value: 0.1911668356033019 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 5, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 372, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8225840845043781, 'min_weight_fraction_leaf': 0.05864804834976011}. Best is trial 46 with value: 0.18593505895462542.
Fold 1 IBS: 0.217351763728157
Fold 2 IBS: 0.16945843188909548
Fold 3 IBS: 0.19034007843019582
Fold 4 IBS: 0.16244329382110362
Fold 5 IBS: 0.19781268762530857
[I 2024-04-16 10:25:55,182] Trial 91 finished with value: 0.1874812510987721 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 19, 'n_estimators': 316, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.88615098

In [86]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [87]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.867
train_ibs:  0.186


#### Test

In [88]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [89]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=14, max_features='auto', max_leaf_nodes=18,
                   max_samples=0.7692755291164257, min_samples_leaf=2,
                   min_samples_split=5,
                   min_weight_fraction_leaf=0.006944921267975795,
                   n_estimators=182, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.616


ExtraSurvivalTrees(max_depth=19, max_features=None, max_leaf_nodes=3,
                   max_samples=0.925572429448618, min_samples_leaf=5,
                   min_samples_split=17,
                   min_weight_fraction_leaf=0.08676721083273177,
                   n_estimators=272, random_state=123, warm_start=True)

IBS: 0.214


In [90]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [91]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 10:26:38,389] A new study created in memory with name: no-name-999ec6b9-c88b-478e-82c0-ab211c8f9a67


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:27:09,273] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:27:22,250] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:34:28,960] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.7399996696599189.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:35:14,735] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:43:57,690] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8840318412875596, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.27382248438555523, 'n_estimators': 385, 'criterion': 'squared_error', 'ccp_alpha': 2.0183033060186855, 'min_weight_fraction_leaf': 0.33643534713187806, 'max_features': 'auto', 'min_impurity_decrease': 5.889654690360788e-06, 'validation_fraction': 0.8062622793646869, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 12 with value: 0.7399996696599189.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:44:39,024] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7565765917190008, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.4300954216773497, 'n_estimators': 440, 'criterion': 'friedma

...

Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 11:55:44,229] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.36065535241543395, 'n_estimators': 435, 'criterion': 'squared_error', 'ccp_alpha': 1.5126136571072866, 'min_weight_fraction_leaf': 0.2002957617254777, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.8814649292460881, 'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 9 with value: 0.2155570193258262.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-16 11:56:04,611] Trial 56 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.2818850250060197, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.23297170241739207, 'n_estimators': 3

Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 12:00:30,741] Trial 66 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9998769833869746, 'learning_rate': 0.003967598379054899, 'dropout_rate': 0.17654958266634313, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.8981368148483696, 'min_weight_fraction_leaf': 0.03518122515344503, 'max_features': 'auto', 'min_impurity_decrease': 7.140027633149786e-05, 'validation_fraction': 0.6362228335648394, 'min_samples_split': 17, 'max_leaf_nodes': 15, 'min_samples_leaf': 18, 'max_depth': 1}. Best is trial 62 with value: 0.21546146572568672.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-16 12:00:55,464] Trial 67 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.8945386029541793, 'learning_rate': 0.01494075298406419, 'dropout_rate': 0.209071801122636, 'n_estimators': 4

Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 12:06:01,516] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9277566139958084, 'learning_rate': 0.05776064499915696, 'dropout_rate': 0.19520203577374073, 'n_estimators': 410, 'criterion': 'squared_error', 'ccp_alpha': 1.2832467794713243, 'min_weight_fraction_leaf': 0.27180012569333806, 'max_features': 1, 'min_impurity_decrease': 6.5181445049333515e-06, 'validation_fraction': 0.6076036057733984, 'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 17, 'max_depth': 1}. Best is trial 75 with value: 0.2064912784368552.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609574
[I 2024-04-16 12:06:20,842] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9641397901643791, 'learning_rate': 0.08426314282635561, 'dropout_rate': 0.1247743198402547, 'n_estimators': 300, 'cr

Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 12:11:30,271] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9384285478087404, 'learning_rate': 0.008432408233149245, 'dropout_rate': 0.2555430411709441, 'n_estimators': 351, 'criterion': 'squared_error', 'ccp_alpha': 0.5917857287890423, 'min_weight_fraction_leaf': 0.30147465751502, 'max_features': 'auto', 'min_impurity_decrease': 1.3322358120453633e-06, 'validation_fraction': 0.7530800224575359, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'max_depth': 2}. Best is trial 75 with value: 0.2064912784368552.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 12:12:09,610] Trial 89 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8566436667378682, 'learning_rate': 0.01150450588881658, 'dropout_rate': 0.1670492025845379, 'n_estimators': 425,

Fold 5 IBS: 0.2170864632573697
[I 2024-04-16 12:25:40,404] Trial 99 finished with value: 0.21448885544366933 and parameters: {'subsample': 0.8012390519326651, 'learning_rate': 0.05269659590897164, 'dropout_rate': 0.27553624278288036, 'n_estimators': 417, 'criterion': 'squared_error', 'ccp_alpha': 0.004383883527848379, 'min_weight_fraction_leaf': 0.353628504086837, 'max_features': 0.1, 'min_impurity_decrease': 8.641052711851521e-07, 'validation_fraction': 0.6285370472109972, 'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 75 with value: 0.2064912784368552.


* Best trial for IBS: 
 FrozenTrial(number=75, state=TrialState.COMPLETE, values=[0.2064912784368552], datetime_start=datetime.datetime(2024, 4, 16, 12, 4, 46, 256407), datetime_complete=datetime.datetime(2024, 4, 16, 12, 5, 14, 629775), params={'subsample': 0.8646743646205938, 'learning_rate': 0.08843409287484169, 'dropout_rate': 0.1693875032679634, 'n_estimators': 403, 'criteri

In [92]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [93]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.74
train_ibs:  0.206


#### Test

In [94]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [95]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.058133567268478875,
                                 criterion='squared_error',
                                 dropout_rate=0.2527776031497929,
                                 learning_rate=0.0012227187192111223,
                                 max_depth=1, max_features='auto',
                                 max_leaf_nodes=20,
                                 min_impurity_decrease=1.5094636128058346e-06,
                                 min_samples_leaf=15, min_samples_split=20,
                                 min_weight_fraction_leaf=0.2151989029549357,
                                 n_estimators=489, random_state=123,
                                 subsample=0.873850481285158,
                                 validation_fraction=0.9945176333416724)

C-index score: 0.618


GradientBoostingSurvivalAnalysis(ccp_alpha=0.008810810992982535,
                                 criterion='squared_error',
                                 dropout_rate=0.1693875032679634,
                                 learning_rate=0.08843409287484169,
                                 max_features='auto', max_leaf_nodes=15,
                                 min_impurity_decrease=5.157320437854681e-06,
                                 min_samples_leaf=16, min_samples_split=20,
                                 min_weight_fraction_leaf=0.29044798941528627,
                                 n_estimators=403, random_state=123,
                                 subsample=0.8646743646205938,
                                 validation_fraction=0.7489142536117352)

IBS: 0.215


In [96]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [97]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [98]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 12:26:20,799] A new study created in memory with name: no-name-08374a21-617b-4f92-a4ff-c5835938cf01


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 12:26:23,111] Trial 0 finished with value: 0.6925028294272052 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6925028294272052.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.75
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 12:26:41,631] Trial 1 finished with value: 0.6915224372703425 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6925028294272052.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.73708920

Fold 1 C-index: 0.5670995670995671
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.679324894514768
Fold 5 C-index: 0.7417840375586855
[I 2024-04-16 12:39:12,350] Trial 19 finished with value: 0.6978167698626153 and parameters: {'subsample': 0.2202898915217226, 'dropout_rate': 0.3156718779160812, 'n_estimators': 431, 'learning_rate': 0.01563167261765183}. Best is trial 14 with value: 0.7072701373860618.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6751054852320675
Fold 5 C-index: 0.7464788732394366
[I 2024-04-16 12:39:20,593] Trial 20 finished with value: 0.6988588248391951 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.22438470567364258, 'n_estimators': 245, 'learning_rate': 0.03827516477850919}. Best is trial 14 with value: 0.7072701373860618.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7647058823529411


Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6877637130801688
Fold 5 C-index: 0.7417840375586855
[I 2024-04-16 12:42:59,553] Trial 38 finished with value: 0.6995315898527518 and parameters: {'subsample': 0.23557094034920928, 'dropout_rate': 0.1525802414016249, 'n_estimators': 46, 'learning_rate': 0.06267653306005645}. Best is trial 21 with value: 0.7100870387945125.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7417840375586855
[I 2024-04-16 12:43:18,500] Trial 39 finished with value: 0.6960685274125858 and parameters: {'subsample': 0.4137092791396724, 'dropout_rate': 0.29699687311164197, 'n_estimators': 384, 'learning_rate': 0.01715036845928685}. Best is trial 21 with value: 0.7100870387945125.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6666666666

...

Fold 2 IBS: 0.17879461279451767
Fold 3 IBS: 0.17573163203218803
Fold 4 IBS: 0.19119338501096567
Fold 5 IBS: 0.18498574070536566
[I 2024-04-16 13:14:03,372] Trial 19 finished with value: 0.1865662636348538 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.1865662636348538.
Fold 1 IBS: 0.20385002842262298
Fold 2 IBS: 0.17898541076661173
Fold 3 IBS: 0.177518113027334
Fold 4 IBS: 0.1979575811954832
Fold 5 IBS: 0.18783103572217225
[I 2024-04-16 13:14:05,611] Trial 20 finished with value: 0.18922843382684484 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.1865662636348538.
Fold 1 IBS: 0.20107429217460354
Fold 2 IBS: 0.186025328567609
Fold 3 IBS: 0.18218745670213524
Fold 4 IBS: 0.20090706949452952
Fold 5 IBS: 0.19364926572166022
[I 2024-04-16

Fold 3 IBS: 0.17687063215564577
Fold 4 IBS: 0.19142503805165928
Fold 5 IBS: 0.17771887919295226
[I 2024-04-16 13:49:29,771] Trial 38 finished with value: 0.18440251539569572 and parameters: {'subsample': 0.10045702238474481, 'dropout_rate': 0.13272164755980653, 'n_estimators': 82, 'learning_rate': 0.0684395247111694}. Best is trial 38 with value: 0.18440251539569572.
Fold 1 IBS: 0.23012380534879587
Fold 2 IBS: 0.17206085972010163
Fold 3 IBS: 0.17890276059875942
Fold 4 IBS: 0.20856260698376405
Fold 5 IBS: 0.17199961045343268
[I 2024-04-16 13:49:33,625] Trial 39 finished with value: 0.19232992862097073 and parameters: {'subsample': 0.3132620855006216, 'dropout_rate': 0.11471626893495929, 'n_estimators': 77, 'learning_rate': 0.06892741183938003}. Best is trial 38 with value: 0.18440251539569572.
Fold 1 IBS: 0.2621725369284015
Fold 2 IBS: 0.26403894140234485
Fold 3 IBS: 0.20945772199901574
Fold 4 IBS: 0.2737981466224617
Fold 5 IBS: 0.21418882926872254
[I 2024-04-16 13:49:39,490] Trial 40 f

Fold 3 IBS: 0.1720321274763833
Fold 4 IBS: 0.20269242936659473
Fold 5 IBS: 0.17347750016357924
[I 2024-04-16 13:51:04,355] Trial 57 finished with value: 0.1894285722337641 and parameters: {'subsample': 0.22308046590336078, 'dropout_rate': 0.16382075159292628, 'n_estimators': 54, 'learning_rate': 0.07586937026396535}. Best is trial 38 with value: 0.18440251539569572.
Fold 1 IBS: 0.24723459364971187
Fold 2 IBS: 0.2044357530332651
Fold 3 IBS: 0.20292788668222125
Fold 4 IBS: 0.223528759028567
Fold 5 IBS: 0.1832818904315015
[I 2024-04-16 13:51:10,653] Trial 58 finished with value: 0.21228177656505337 and parameters: {'subsample': 0.2849097248903497, 'dropout_rate': 0.12541109823464971, 'n_estimators': 133, 'learning_rate': 0.06314379319286169}. Best is trial 38 with value: 0.18440251539569572.
Fold 1 IBS: 0.22470401483207397
Fold 2 IBS: 0.18044827895656998
Fold 3 IBS: 0.1770853252772186
Fold 4 IBS: 0.22051345390766136
Fold 5 IBS: 0.17624589843250807
[I 2024-04-16 13:51:14,890] Trial 59 fini

Fold 2 IBS: 0.17136294720501474
Fold 3 IBS: 0.1942205278629809
Fold 4 IBS: 0.18836548573686962
Fold 5 IBS: 0.18371776191244502
[I 2024-04-16 13:52:47,423] Trial 76 finished with value: 0.19432330597275116 and parameters: {'subsample': 0.12565758748945094, 'dropout_rate': 0.1008354138446842, 'n_estimators': 115, 'learning_rate': 0.06744663662796715}. Best is trial 72 with value: 0.1836153294988586.
Fold 1 IBS: 0.2122887185855283
Fold 2 IBS: 0.18559325172580513
Fold 3 IBS: 0.17394637292345108
Fold 4 IBS: 0.21069765756694586
Fold 5 IBS: 0.18280497330673373
[I 2024-04-16 13:52:49,610] Trial 77 finished with value: 0.1930661948216928 and parameters: {'subsample': 0.549746960211358, 'dropout_rate': 0.18609046599680273, 'n_estimators': 36, 'learning_rate': 0.06375434111629431}. Best is trial 72 with value: 0.1836153294988586.
Fold 1 IBS: 0.2368850951432027
Fold 2 IBS: 0.2028084971050872
Fold 3 IBS: 0.1912826057200971
Fold 4 IBS: 0.2040995320657804
Fold 5 IBS: 0.180053379848087
[I 2024-04-16 1

Fold 3 IBS: 0.1786201819401681
Fold 4 IBS: 0.1730106628597333
Fold 5 IBS: 0.1769335760691037
[I 2024-04-16 13:53:44,020] Trial 95 finished with value: 0.18322424683870203 and parameters: {'subsample': 0.10113592213603971, 'dropout_rate': 0.12650276574260835, 'n_estimators': 75, 'learning_rate': 0.07005155964295467}. Best is trial 95 with value: 0.18322424683870203.
Fold 1 IBS: 0.24361216774620978
Fold 2 IBS: 0.18983481001075334
Fold 3 IBS: 0.21205828714477906
Fold 4 IBS: 0.1712402193849143
Fold 5 IBS: 0.19545836210091558
[I 2024-04-16 13:53:51,088] Trial 96 finished with value: 0.2024407692775144 and parameters: {'subsample': 0.10436272208850539, 'dropout_rate': 0.12665776901745035, 'n_estimators': 134, 'learning_rate': 0.08154732519464149}. Best is trial 95 with value: 0.18322424683870203.
Fold 1 IBS: 0.2261988766991688
Fold 2 IBS: 0.17895050409910299
Fold 3 IBS: 0.17889485274253647
Fold 4 IBS: 0.2076135194010375
Fold 5 IBS: 0.1758505273629191
[I 2024-04-16 13:53:55,174] Trial 97 fini

In [99]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [100]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.71
train_ibs:  0.183


#### Test

In [101]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [102]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3058920522870888,
                                              learning_rate=0.0016108337344105245,
                                              n_estimators=410,
                                              random_state=123,
                                              subsample=0.11014863416226671)

C-index score: 0.576


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.12650276574260835,
                                              learning_rate=0.07005155964295467,
                                              n_estimators=75, random_state=123,
                                              subsample=0.10113592213603971)

IBS: 0.246


In [103]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [104]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.871,1.0
ExtraSurvivalTrees,0.867,2.0
GradientBoosting,0.740,3.0
ComponentwiseGradientBoosting,0.710,4.0
CoxElastic,0.708,5.0
CoxRidge,0.698,6.0
CoxLasso,0.541,7.0


In [105]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.183,1.5
ComponentwiseGradientBoosting,0.183,1.5
ExtraSurvivalTrees,0.186,3.0
GradientBoosting,0.206,4.0
CoxElastic,0.212,5.0
CoxRidge,0.217,6.0
CoxLasso,0.392,7.0


In [106]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.628,1.0
GradientBoosting,0.618,2.0
ExtraSurvivalTrees,0.616,3.0
CoxRidge,0.586,4.5
CoxElastic,0.586,4.5
ComponentwiseGradientBoosting,0.576,6.0
CoxLasso,0.500,7.0


In [107]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ExtraSurvivalTrees,0.214,1.0
GradientBoosting,0.215,2.0
CoxRidge,0.221,3.5
CoxElastic,0.221,3.5
Randomsurvivalforest,0.226,5.0
ComponentwiseGradientBoosting,0.246,6.0
CoxLasso,0.430,7.0


In [108]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/os/standard/no_selection/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_os_standard_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file

from datetime import date

current_date = date.today()
print(current_date)